<div align="center">

<!-- Judul Laporan -->
<h1><strong>KECERDASAN ARTIFISIAL UNTUK GEOFISIKA</strong></h1>
</ul>
<!-- Logo Universitas -->
<img src="https://www.unhas.ac.id/wp-content/uploads/2022/09/Logo-Resmi-Unhas-1.png" 
     alt="Logo Universitas Hasanuddin" 
     style="width:250px; margin:20px 0;">

<div align="center">

<!-- Penulis -->
<h3>Disusun Oleh:</h3>
<ul style="list-style:none; padding-left:0; line-height:1.8; width:400px;">
  <li style="display:flex; justify-content:space-between;"><span>MUHAMMAD RIDWAN</span><span>(H061231002)</span></li>
</ul>
<br><br>
<!-- Departemen dan Fakultas -->
<p><strong>PRODI GEOFISIKA</strong></p>
<p><strong>FAKULTAS MATEMATIKA DAN ILMU PENGETAHUAN ALAM</strong></p>
<p><strong>UNIVERSITAS HASANUDDIN</strong></p>
<p><strong>MAKASSAR</strong></p>

<!-- Tahun -->
<p><strong>2026</strong></p>

</div>
  

### Peta elevasi DEM, kemiringan lereng, aspect, curah hujan, dan jarak ke sungai.
### Peta indeks kerentanan longsor relatif (Landslide Susceptibility Index / LSI).
### Peta klasifikasi kerentanan.
### Raster GeoTIFF setiap parameter dan hasil akhir.
### Tabel statistik luas setiap kelas dalam hektare dan persentase.
### Catatan konfigurasi, metadata input, serta laporan interpretasi.

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
R-GIS-001 | Pemetaan Kerentanan Longsor Berbasis GIS dengan Weighted Overlay

PENTING:
- Workflow ini membaca data input yang disediakan pengguna; tidak membuat data lapangan.
- Tidak mengarang CRS, parameter yang tidak tersedia, atau skor kategori tak dikenal.
- Hasil adalah indeks kerentanan relatif (LSI), bukan probabilitas longsor atau faktor keamanan.
- Input dipertahankan; output ditulis ke folder run baru.
- Skor/bobot default adalah CONTOH DEMONSTRASI dan wajib ditinjau sebelum penggunaan nyata.
"""

from __future__ import annotations
import argparse
import json
import sys
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import rasterize
import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from scipy.ndimage import distance_transform_edt


# ----------------------------- CONFIGURATION -----------------------------
DEFAULT_WEIGHTS = {
    "slope": 0.30,
    "lithology": 0.25,
    "lulc": 0.15,
    "rainfall": 0.20,
    "distance_to_river": 0.10,
}
CLASS_NAMES = {1: "Rendah", 2: "Sedang", 3: "Tinggi", 4: "Sangat Tinggi"}
CLASS_COLORS = ["#2ca25f", "#fee08b", "#fdae61", "#d73027"]


def parse_args(argv=None):
    """Parse argumen CLI; aman juga saat dipanggil dari Jupyter Notebook."""
    p = argparse.ArgumentParser(description="Recipe R-GIS-001")
    p.add_argument(
        "--config", type=Path, default=Path("config.json"),
        help="Path ke konfigurasi JSON"
    )
    p.add_argument(
        "--run-id", default=None,
        help="Nama run; default timestamp"
    )
    # parse_known_args mengabaikan argumen internal kernel Jupyter seperti --f=...
    args, unknown = p.parse_known_args(argv)
    if unknown:
        print("Argumen tambahan diabaikan:", unknown)
    return args


def load_config(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Config tidak ditemukan: {path.resolve()}\n"
            "Salin template config.json dan isi path input yang benar."
        )
    cfg = json.loads(path.read_text(encoding="utf-8"))
    required = ["aoi", "dem", "lithology", "lulc", "rainfall", "rivers",
                "analysis_crs", "target_resolution_m", "weights", "classes"]
    missing = [k for k in required if k not in cfg]
    if missing:
        raise ValueError(f"Field config wajib tidak ada: {missing}")
    return cfg


def check_weights(weights: dict):
    needed = ["slope", "lithology", "lulc", "rainfall", "distance_to_river"]
    absent = [k for k in needed if k not in weights]
    if absent:
        raise ValueError(f"Bobot parameter wajib belum ada: {absent}")
    vals = [float(weights[k]) for k in needed]
    if any(not np.isfinite(v) or v < 0 for v in vals):
        raise ValueError("Bobot harus finite dan tidak negatif.")
    if not np.isclose(sum(vals), 1.0, atol=1e-6):
        raise ValueError(f"Jumlah bobot harus 1.00; sekarang {sum(vals):.8f}")


def read_vector(path: str, layer: str | None, target_crs: str, label: str):
    path_obj = Path(path)
    if not path_obj.exists():
        raise FileNotFoundError(f"File vektor {label} tidak ditemukan: {path_obj}")
    kwargs = {"layer": layer} if layer else {}
    gdf = gpd.read_file(path_obj, **kwargs)
    if gdf.crs is None:
        raise ValueError(
            f"CRS {label} tidak tersedia. Jangan menebak CRS; tetapkan metadata yang benar."
        )
    if gdf.empty:
        raise ValueError(f"Layer {label} kosong.")
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    if gdf.empty:
        raise ValueError(f"Layer {label} tidak memiliki geometri valid/nonkosong.")
    # Repair is deliberately not automatic; invalid geometry must be inspected.
    invalid_count = int((~gdf.geometry.is_valid).sum())
    if invalid_count:
        raise ValueError(
            f"{label}: ditemukan {invalid_count} geometri invalid. Periksa/perbaiki salinan "
            "data secara eksplisit sebelum menjalankan recipe."
        )
    return gdf.to_crs(target_crs)


def read_raster_to_grid(path: str, dst_crs: str, bounds, resolution: float,
                        resampling: Resampling, is_categorical=False):
    """Reproject/resample one raster to a shared target grid."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Raster tidak ditemukan: {p}")
    with rasterio.open(p) as src:
        if src.crs is None:
            raise ValueError(f"Raster {p.name} tidak memiliki CRS; jangan menebak.")
        if src.count < 1:
            raise ValueError(f"Raster {p.name} tidak memiliki band.")
        left, bottom, right, top = bounds
        width = int(np.ceil((right - left) / resolution))
        height = int(np.ceil((top - bottom) / resolution))
        if width <= 0 or height <= 0:
            raise ValueError("Grid target tidak valid; periksa AOI, CRS, dan resolusi.")
        transform = rasterio.transform.from_origin(left, top, resolution, resolution)
        dst = np.full((height, width), np.nan, dtype="float32")
        src_nodata = src.nodata
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=resampling
        )
        profile = {
            "driver": "GTiff", "height": height, "width": width, "count": 1,
            "dtype": "float32", "crs": dst_crs, "transform": transform,
            "nodata": -9999.0, "compress": "lzw"
        }
    return dst, profile


def rasterize_vector(gdf, value_field, shape, transform, categorical_map=None):
    shapes = []
    if value_field not in gdf.columns:
        raise ValueError(f"Field '{value_field}' tidak ada. Field tersedia: {list(gdf.columns)}")
    for _, row in gdf.iterrows():
        val = row[value_field]
        if categorical_map is not None:
            if val not in categorical_map:
                continue
            val = categorical_map[val]
        try:
            val = float(val)
        except (TypeError, ValueError):
            continue
        if np.isfinite(val):
            shapes.append((row.geometry, val))
    if not shapes:
        raise ValueError(
            f"Tidak ada fitur {value_field} yang dapat dirasterisasi. "
            "Periksa nama field dan mapping kategori."
        )
    arr = rasterize(
        shapes, out_shape=shape, transform=transform, fill=np.nan,
        dtype="float32", all_touched=False
    )
    # Rasterio rasterize fill for float NaN is supported.
    return arr


def calculate_slope_aspect(dem, resolution):
    # DEM elevations must be in metres if horizontal CRS is metres.
    # np.gradient cannot reliably operate across NaN boundaries, so fill NaNs
    # temporarily with nearest finite cells; mask is restored after derivatives.
    valid = np.isfinite(dem)
    if valid.sum() < 9:
        raise ValueError("Sel raster DEM valid terlalu sedikit untuk menghitung slope.")
    # nearest-valid fill for derivative stability at AOI edges
    from scipy.ndimage import distance_transform_edt as edt
    indices = edt(~valid, return_distances=False, return_indices=True)
    filled = dem[tuple(indices)]
    gy, gx = np.gradient(filled, resolution, resolution)
    slope = np.degrees(np.arctan(np.sqrt(gx * gx + gy * gy)))
    aspect = (np.degrees(np.arctan2(-gx, gy)) + 360.0) % 360.0
    slope[~valid] = np.nan
    aspect[~valid] = np.nan
    return slope, aspect


def reclassify_slope(slope, bins):
    # bins are upper bounds, final class includes values above the last bound
    arr = np.full(slope.shape, np.nan, dtype="float32")
    valid = np.isfinite(slope)
    edges = [float(x) for x in bins]
    if sorted(edges) != edges or len(edges) != 4:
        raise ValueError("slope_upper_bounds_deg harus berisi 4 angka meningkat untuk skor 1-5.")
    arr[valid] = np.digitize(slope[valid], edges, right=True) + 1
    return arr


def reclassify_categorical(raster, mapping, label):
    arr = np.full(raster.shape, np.nan, dtype="float32")
    valid = np.isfinite(raster)
    known = np.zeros(raster.shape, dtype=bool)
    for code, score in mapping.items():
        mask_ = valid & (raster == float(code))
        arr[mask_] = float(score)
        known |= mask_
    unknown = valid & ~known
    if unknown.any():
        values = np.unique(raster[unknown]).tolist()
        raise ValueError(
            f"Kode kategori tak dikenal pada {label}: {values}. "
            "Tambahkan mapping eksplisit ke config; tidak diberi skor otomatis."
        )
    return arr


def reclassify_continuous(raster, breaks, label):
    """Four increasing boundaries produce five ordered classes, scores 1-5."""
    edges = [float(x) for x in breaks]
    if len(edges) != 4 or sorted(edges) != edges:
        raise ValueError(f"{label}: dibutuhkan empat batas meningkat untuk lima kelas.")
    out = np.full(raster.shape, np.nan, dtype="float32")
    valid = np.isfinite(raster)
    out[valid] = np.digitize(raster[valid], edges, right=True) + 1
    return out


def reclassify_distance_near_is_high(distance, breaks):
    """Distance classes are ascending in metres; nearer river gets higher score."""
    edges = [float(x) for x in breaks]
    if len(edges) != 4 or sorted(edges) != edges:
        raise ValueError("river_distance_breaks_m harus empat batas meningkat.")
    out = np.full(distance.shape, np.nan, dtype="float32")
    valid = np.isfinite(distance)
    # distance <= edge[0] -> score 5; increasing distance -> lower score
    rank = np.digitize(distance[valid], edges, right=True) + 1
    out[valid] = 6 - rank
    return out


def calc_distance_to_river(river_gdf, shape, transform, resolution):
    river_mask = rasterize(
        [(geom, 1) for geom in river_gdf.geometry],
        out_shape=shape, transform=transform, fill=0, dtype="uint8"
    ).astype(bool)
    if not river_mask.any():
        raise ValueError("Sungai tidak mengenai grid target; periksa extent/CRS.")
    # distance_transform_edt calculates distance to nearest zero; zeros are river cells.
    distance = distance_transform_edt(~river_mask, sampling=(resolution, resolution))
    distance[~np.isfinite(distance)] = np.nan
    return distance.astype("float32")


def save_raster(path, arr, profile, description):
    output = np.where(np.isfinite(arr), arr, profile.get("nodata", -9999)).astype("float32")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(output, 1)
        dst.set_band_description(1, description)
        dst.update_tags(
            recipe_id="R-GIS-001",
            data_status="derived_from_user_inputs",
            interpretation="Relative susceptibility index; not landslide probability."
        )


def plot_map(arr, profile, title, output_path, cmap="viridis",
             cbar_label="", vmin=None, vmax=None, categorical=False, labels=None):
    transform = profile["transform"]
    height, width = arr.shape
    left = transform.c
    top = transform.f
    right = left + width * transform.a
    bottom = top + height * transform.e
    fig, ax = plt.subplots(figsize=(9, 7))
    if categorical:
        cm = ListedColormap(CLASS_COLORS if labels is None else cmap)
        norm = BoundaryNorm(np.arange(0.5, 5.5, 1), cm.N)
        im = ax.imshow(
            arr, extent=(left, right, bottom, top), origin="upper",
            cmap=cm, norm=norm
        )
        cbar = fig.colorbar(im, ax=ax, ticks=[1, 2, 3, 4], shrink=0.78)
        cbar.ax.set_yticklabels(["Rendah", "Sedang", "Tinggi", "Sangat Tinggi"])
    else:
        im = ax.imshow(
            arr, extent=(left, right, bottom, top), origin="upper",
            cmap=cmap, vmin=vmin, vmax=vmax
        )
        cbar = fig.colorbar(im, ax=ax, shrink=0.78)
        cbar.set_label(cbar_label)
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel(f"Easting (m) — {profile['crs']}")
    ax.set_ylabel("Northing (m)")
    ax.set_aspect("equal")
    ax.grid(True, linestyle=":", alpha=0.25)
    ax.annotate(
        "N", xy=(0.94, 0.88), xytext=(0.94, 0.78),
        xycoords="axes fraction", ha="center", va="center",
        arrowprops=dict(arrowstyle="-|>", lw=1.5), fontsize=12
    )
    # Simple graphic scale bar of 10% map width (not fixed 500 m).
    bar_len = (right - left) * 0.10
    bar_x = left + (right - left) * 0.05
    bar_y = bottom + (top - bottom) * 0.06
    ax.plot([bar_x, bar_x + bar_len], [bar_y, bar_y], color="black", linewidth=3)
    ax.text(bar_x + bar_len / 2, bar_y + (top-bottom)*0.02,
            f"{bar_len:.0f} m", ha="center", fontsize=8,
            bbox=dict(facecolor="white", alpha=0.75, edgecolor="none"))
    fig.text(0.01, 0.01, "Recipe R-GIS-001 | Indeks relatif | Verifikasi sumber/tahun sebelum publikasi",
             fontsize=7)
    fig.tight_layout(rect=(0, 0.035, 1, 1))
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


def main():
    args = parse_args()
    config_path = args.config.resolve()
    cfg = load_config(config_path)
    check_weights(cfg["weights"])

    target_crs = cfg["analysis_crs"]
    resolution = float(cfg["target_resolution_m"])
    if resolution <= 0:
        raise ValueError("target_resolution_m harus > 0.")
    run_id = args.run_id or datetime.now().strftime("%Y%m%d_%H%M%S")
    output_root = Path(cfg.get("output_dir", "results/gis")).resolve()
    run_dir = output_root / run_id
    if run_dir.exists():
        raise FileExistsError(f"Folder run sudah ada: {run_dir}; gunakan --run-id berbeda.")
    (run_dir / "maps").mkdir(parents=True)
    (run_dir / "rasters").mkdir()
    (run_dir / "tables").mkdir()

    started = datetime.now(timezone.utc).isoformat()
    warnings = []
    try:
        aoi = read_vector(cfg["aoi"]["path"], cfg["aoi"].get("layer"), target_crs, "AOI")
        # Union AOI geometry, projected before calculating bounds.
        aoi_geom = aoi.geometry.union_all()
        bounds = aoi_geom.bounds

        # Validate required vectors and reproject them.
        lith = read_vector(cfg["lithology"]["path"], cfg["lithology"].get("layer"),
                           target_crs, "litologi")
        lulc_vec = cfg["lulc"].get("type", "raster") == "vector"
        if lulc_vec:
            lulc_gdf = read_vector(cfg["lulc"]["path"], cfg["lulc"].get("layer"),
                                   target_crs, "LULC")
        rivers = read_vector(cfg["rivers"]["path"], cfg["rivers"].get("layer"),
                             target_crs, "sungai")

        dem, profile = read_raster_to_grid(
            cfg["dem"]["path"], target_crs, bounds, resolution, Resampling.bilinear
        )
        # Rasterize AOI and mask DEM outside polygon.
        inside = rasterize(
            [(aoi_geom, 1)], out_shape=dem.shape,
            transform=profile["transform"], fill=0, dtype="uint8"
        ).astype(bool)
        dem[~inside] = np.nan

        slope, aspect = calculate_slope_aspect(dem, resolution)

        # Prepare rainfall raster, or stop rather than inventing rainfall values.
        rainfall, _ = read_raster_to_grid(
            cfg["rainfall"]["path"], target_crs, bounds, resolution,
            Resampling.bilinear
        )
        rainfall[~inside] = np.nan

        # Categorical litology: configured field and mapping of exact attribute values to codes.
        lith_field = cfg["lithology"]["class_field"]
        lith_map = cfg["lithology"]["category_to_code"]
        lith_raster = rasterize_vector(
            lith, lith_field, dem.shape, profile["transform"], lith_map
        )
        lith_raster[~inside] = np.nan

        if lulc_vec:
            lulc_field = cfg["lulc"]["class_field"]
            lulc_map = cfg["lulc"]["category_to_code"]
            lulc_raster = rasterize_vector(
                lulc_gdf, lulc_field, dem.shape, profile["transform"], lulc_map
            )
        else:
            lulc_raster, _ = read_raster_to_grid(
                cfg["lulc"]["path"], target_crs, bounds, resolution,
                Resampling.nearest
            )
        lulc_raster[~inside] = np.nan

        river_dist = calc_distance_to_river(
            rivers, dem.shape, profile["transform"], resolution
        )
        river_dist[~inside] = np.nan

        score_cfg = cfg["scores"]
        slope_score = reclassify_slope(slope, score_cfg["slope_upper_bounds_deg"])
        lith_score = reclassify_categorical(
            lith_raster, score_cfg["lithology_code_to_score"], "litologi"
        )
        lulc_score = reclassify_categorical(
            lulc_raster, score_cfg["lulc_code_to_score"], "LULC"
        )
        rainfall_score = reclassify_continuous(
            rainfall, score_cfg["rainfall_breaks"], "curah hujan"
        )
        river_score = reclassify_distance_near_is_high(
            river_dist, score_cfg["river_distance_breaks_m"]
        )

        score_arrays = {
            "slope": slope_score,
            "lithology": lith_score,
            "lulc": lulc_score,
            "rainfall": rainfall_score,
            "distance_to_river": river_score
        }

        # Conservative NoData policy: calculate LSI only where every weighted input exists.
        all_valid = inside.copy()
        for name, arr in score_arrays.items():
            all_valid &= np.isfinite(arr)
            save_raster(run_dir / "rasters" / f"score_{name}.tif",
                        arr, profile, f"Skor {name}")
        lsi = np.full(dem.shape, np.nan, dtype="float32")
        lsi[all_valid] = sum(
            float(cfg["weights"][name]) * score_arrays[name][all_valid]
            for name in score_arrays
        )

        class_cfg = cfg["classes"]
        class_edges = [float(x) for x in class_cfg["lsi_upper_bounds"]]
        if len(class_edges) != 3 or sorted(class_edges) != class_edges:
            raise ValueError("lsi_upper_bounds harus tiga batas meningkat untuk empat kelas.")
        classes = np.full(dem.shape, np.nan, dtype="float32")
        classes[all_valid] = np.digitize(
            lsi[all_valid], class_edges, right=True
        ) + 1

        derived = {
            "dem_m": dem,
            "slope_deg": slope,
            "aspect_deg": aspect,
            "rainfall": rainfall,
            "lithology_code": lith_raster,
            "lulc_code": lulc_raster,
            "distance_to_river_m": river_dist,
            "LSI": lsi,
            "susceptibility_class": classes
        }
        for name, arr in derived.items():
            save_raster(run_dir / "rasters" / f"{name}.tif", arr, profile, name)

        # Maps: source year/source citation should be added to config/report before publication.
        plot_map(dem, profile, "Elevasi DEM", run_dir / "maps/elevation.png",
                 cmap="terrain", cbar_label="Elevasi (unit source; pastikan meter)")
        plot_map(slope, profile, "Kemiringan Lereng", run_dir / "maps/slope.png",
                 cmap="YlOrRd", cbar_label="Derajat", vmin=0, vmax=60)
        plot_map(aspect, profile, "Aspect", run_dir / "maps/aspect.png",
                 cmap="twilight", cbar_label="Derajat", vmin=0, vmax=360)
        plot_map(rainfall, profile, "Curah Hujan", run_dir / "maps/rainfall.png",
                 cmap="Blues", cbar_label=cfg["rainfall"].get("unit", "unit belum dinyatakan"))
        plot_map(river_dist, profile, "Jarak ke Sungai", run_dir / "maps/distance_to_river.png",
                 cmap="viridis", cbar_label="Meter")
        plot_map(lsi, profile, "Indeks Kerentanan Longsor Relatif (LSI)",
                 run_dir / "maps/LSI.png", cmap="YlOrRd", cbar_label="Indeks LSI")
        plot_map(classes, profile, "Kelas Kerentanan Longsor Relatif",
                 run_dir / "maps/susceptibility_map.png", categorical=True)

        # Inventory metadata.
        inv = []
        for key in ["aoi", "dem", "lithology", "lulc", "rainfall", "rivers"]:
            item = cfg[key]
            fp = Path(item["path"])
            entry = {"dataset_key": key, "path": str(fp), "exists": fp.exists(),
                     "declared_source": item.get("source", "tidak dicantumkan"),
                     "year": item.get("year", "tidak dicantumkan")}
            if fp.exists() and key not in ["dem", "rainfall"] and item.get("type") != "raster":
                try:
                    gdf = gpd.read_file(fp, layer=item.get("layer")) if item.get("layer") else gpd.read_file(fp)
                    entry["format"] = fp.suffix.lower()
                    entry["crs_source"] = str(gdf.crs)
                    entry["feature_count"] = len(gdf)
                except Exception as e:
                    entry["inventory_note"] = str(e)
            elif fp.exists():
                try:
                    with rasterio.open(fp) as src:
                        entry.update({"format": fp.suffix.lower(), "crs_source": str(src.crs),
                                      "resolution_source": list(src.res),
                                      "width": src.width, "height": src.height,
                                      "nodata": src.nodata})
                except Exception as e:
                    entry["inventory_note"] = str(e)
            inv.append(entry)
        pd.DataFrame(inv).to_csv(run_dir / "tables/inventory_data.csv", index=False)

        # Area statistics: projected CRS unit should be metres for this calculation.
        crs_obj = rasterio.crs.CRS.from_user_input(target_crs)
        if not crs_obj.is_projected:
            raise ValueError("CRS analisis harus CRS proyeksi untuk menghitung luas.")
        cell_area = abs(profile["transform"].a * profile["transform"].e)
        valid_cells = int(np.isfinite(classes).sum())
        rows = []
        for code, name in CLASS_NAMES.items():
            n = int(np.count_nonzero(classes == code))
            rows.append({
                "class_id": code, "class_name": name, "cell_count": n,
                "area_m2": n * cell_area, "area_ha": n * cell_area / 10000,
                "area_percent_valid": 100 * n / valid_cells if valid_cells else 0,
                "lsi_lower_bound": [None, class_edges[0], class_edges[1], class_edges[2]][code-1],
                "lsi_upper_bound": [class_edges[0], class_edges[1], class_edges[2], None][code-1]
            })
        pd.DataFrame(rows).to_csv(run_dir / "tables/area_statistics.csv", index=False)

        # Save reclassification/bobots and explicit limitations.
        report = {
            "recipe_id": "R-GIS-001",
            "prompt_id": "P-GIS-001",
            "status": "Executed locally; requires review of inputs and outputs",
            "started_utc": started,
            "finished_utc": datetime.now(timezone.utc).isoformat(),
            "python": platform.python_version(),
            "analysis_crs": target_crs,
            "resolution_m": resolution,
            "weights": cfg["weights"],
            "weight_sum": float(sum(cfg["weights"].values())),
            "scores": score_cfg,
            "class_edges": class_edges,
            "valid_lsi_cells": int(np.isfinite(lsi).sum()),
            "aoi_grid_cells": int(inside.sum()),
            "nodata_cells_within_aoi": int(inside.sum() - np.isfinite(classes).sum()),
            "warnings": warnings,
            "limitations": [
                "Output is a weighted-overlay relative index, not a probability.",
                "Output is not a factor of safety or slope-stability analysis.",
                "Weights and scores require local justification and sensitivity/validation checks.",
                "Aspect is derived and mapped but is not included in the weighted overlay.",
                "The output is only as reliable as source scale, resolution, units, and metadata."
            ]
        }
        (run_dir / "config_and_run_record.json").write_text(
            json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        summary_lines = [
            "# Interpretasi awal — R-GIS-001",
            "",
            "**Peringatan:** hasil merupakan indeks kerentanan relatif; bukan probabilitas longsor.",
            "",
            f"- CRS analisis: `{target_crs}`",
            f"- Resolusi target: {resolution:g} m",
            f"- Bobot: `{json.dumps(cfg['weights'], ensure_ascii=False)}`",
            f"- Jumlah sel AOI: {int(inside.sum())}",
            f"- Sel LSI valid: {int(np.isfinite(lsi).sum())}",
            f"- Sel NoData/tidak lengkap di AOI: {int(inside.sum() - np.isfinite(classes).sum())}",
            "",
            "Area kelas tinggi hanya menunjukkan skor relatif menurut parameter, bobot, dan batas kelas yang dipakai. "
            "Periksa peta parameter dan data lapangan sebelum menentukan prioritas investigasi.",
            "",
            "Recipe ini tidak menghitung faktor keamanan dan tidak menggantikan penyelidikan geoteknik."
        ]
        (run_dir / "interpretation_report.md").write_text(
            "\n".join(summary_lines), encoding="utf-8"
        )
        print(f"SELESAI. Output tersimpan di: {run_dir}")
        print(f"Sel LSI valid: {int(np.isfinite(lsi).sum())}/{int(inside.sum())}")
        print("Periksa config_and_run_record.json dan peta sebelum interpretasi.")
    except Exception:
        # Preserve the run directory and error record for diagnosis.
        err = {
            "recipe_id": "R-GIS-001",
            "status": "Failed",
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "error": str(sys.exc_info()[1])
        }
        (run_dir / "failure_record.json").write_text(
            json.dumps(err, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        raise


if __name__ == "__main__":
    main()
